# Voxel intensity distributions (VID) and one-point statistics

Paper figures, from the **production** delta T_b maps (`Tb_maps_prod/H_real`, `Tb_maps_prod/He`; real space).

1. On ORCD: `sbatch run_vid.sh` (set `LINE=H` or `LINE=He` at the top), then `python make_vid.py table --line H --out /orcd/data/mvogelsb/004/chcho/VID/H` (and the same for He).
   `make_vid.py` is the calculation of `calculate_vid_logbins` in `one-stat.ipynb`, unchanged (100 log bins per sign, floor 1e-9, normalised to all voxels).
2. Run the setup cell, then the figure cells. The figure code is that of `one-stat.ipynb` cells 8, 9 (hydrogen) and 31, 32 (helium); only the data sources changed.

`one-stat.ipynb` is kept unchanged as the archive (its VIDs came from earlier map versions).


In [ ]:
import os
import h5py
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
from scipy.integrate import quad
from astropy import constants as const
from scipy.interpolate import interp1d
from scipy.integrate import quad
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200
mpl.rcParams['savefig.dpi'] = 300

import matplotlib as mpl
mpl.rcParams['agg.path.chunksize'] = 10000

# Set universal font properties
plt.rcParams.update({
    "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
    "font.size": 14,
})

import re
import pandas as pd
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# --------------------------------
# Publication-quality style
# --------------------------------

mpl.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,

    # thicker axis box
    "axes.linewidth": 2.0,

    # thicker major/minor ticks
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.minor.width": 1.2,
    "ytick.minor.width": 1.2,

    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "xtick.minor.size": 4,
    "ytick.minor.size": 4,

    # figure quality
    "figure.dpi": 150,
    "savefig.dpi": 300,
})



# =====================================
# Data: production maps (Tb_maps_prod) and their VIDs / statistics (make_vid.py, run_vid.sh)
# =====================================
MAPS = {"H":  "/orcd/data/mvogelsb/004/chcho/Tb_maps_prod/H_real",
        "He": "/orcd/data/mvogelsb/004/chcho/Tb_maps_prod/He"}
VID  = {"H":  "/orcd/data/mvogelsb/004/chcho/VID/H",
        "He": "/orcd/data/mvogelsb/004/chcho/VID/He"}


def get_redshift(ps_file):
    with open(ps_file) as f:
        return float(f.readline().strip())


def load_vid_csv(filename):
    """positive side of a VID file (as in one-stat.ipynb)"""
    df = pd.read_csv(filename)
    df_pos = df[df["side"] == "pos"].copy()
    return df_pos["Tb"].values, df_pos["vid_prob"].values


def load_stats(line, smin, smax):
    """one-point statistics table of make_vid.py (stats_<line>.csv), snapshots smin..smax, high z first"""
    df = pd.read_csv(f"{VID[line]}/stats_{line}.csv")
    return df[(df["snap"] >= smin) & (df["snap"] <= smax)].sort_values("z", ascending=False)


## Hydrogen (21 cm)


### (a) VID at <x_HII> = 0.1 ... 0.9  (one-stat.ipynb cell 8)


In [ ]:
# --------------------------------
# Input files
# --------------------------------

DIR = MAPS["H"]

# the snapshots where the volume-weighted <x_HII> = 0.1 ... 0.9 (check_ionization_history.py)
PS_files = [f"{DIR}/PS_{s:03d}.out" for s in (177, 203, 220, 242, 264, 283, 300, 318, 338)]

def plot_vid_from_csv(PS_files):

    fig, ax = plt.subplots(figsize=(8, 6))

    zs = []
    snaps = []

    # --------------------------------
    # Collect redshifts
    # --------------------------------

    for PS_file in PS_files:
        snap = int(PS_file.split("_")[-1].split(".")[0])
        z = get_redshift(PS_file)

        snaps.append(snap)
        zs.append(z)

    zs = np.array(zs)

    # --------------------------------
    # Colormap
    # --------------------------------

    cmap = plt.get_cmap("plasma_r")
    norm = mpl.colors.Normalize(
        vmin=zs.min(),
        vmax=zs.max()
    )

    # --------------------------------
    # Plot curves
    # --------------------------------

    for snap, z in zip(snaps, zs):

        filename = f"{VID['H']}/VID_H_{snap:03d}.csv"

        Tb, vid = load_vid_csv(filename)

        # avoid log(0)
        vid = np.where(
            vid > 0,
            vid,
            1e-20
        )

        ax.plot(
            Tb,
            vid,
            color=cmap(norm(z)),
            lw=4,
            alpha=0.8
        )

    # --------------------------------
    # Axis scales
    # --------------------------------

    ax.set_xscale(
        "symlog",
        linthresh=0.1
    )

    ax.set_yscale("log")

    # --------------------------------
    # Labels
    # --------------------------------

    ax.set_xlabel(
        r"$\delta T_b^{\mathrm{H}}\ [{\rm mK}]$", fontsize=24
    )

    # Better than just "VID"
    ax.set_ylabel(
        r"$\delta T_b^{\mathrm{H}}$ PDF", fontsize=24
    )

    # --------------------------------
    # Limits
    # --------------------------------

    ax.set_xlim(0.0, 250)
    ax.set_ylim(1e-7, 1e3)

    # --------------------------------
    # Tick style
    # --------------------------------

    ax.tick_params(
        which="both",
        direction="in",
        top=True,
        right=True,
        labelsize=20          # <-- x and y tick numbers
    )

    ax.minorticks_on()

    # --------------------------------
    # Embedded horizontal colorbar
    # --------------------------------

    sm = mpl.cm.ScalarMappable(
        cmap=cmap,
        norm=norm
    )
    sm.set_array([])

    cax = inset_axes(
        ax,
        width="45%",
        height="4%",
        loc="lower left",
        borderpad=4.0
    )

    cbar = fig.colorbar(
        sm,
        cax=cax,
        orientation="horizontal"
    )

    cbar.set_label(
        r"$z$",
        fontsize=20
    )

    cbar.ax.tick_params(
        labelsize=15
    )

    # --------------------------------
    # Exact z ↔ x_HII mapping
    # --------------------------------

    xhii_vals = np.array([
        0.90,
        0.70,
        0.50,
        0.40,
        0.30,
        0.20,
        0.10
    ])

    z_match = np.array([
        5.826679,
        6.342011,
        6.872631,
        7.204283,
        7.624910,
        8.207879,
        9.170129
    ])

    # bottom ticks = z
    cbar.set_ticks(z_match)
    cbar.set_ticklabels(
        [f"{z:.1f}" for z in z_match]
    )

    # high-z on left
    cbar.ax.invert_xaxis()

    # --------------------------------
    # Top axis = ionized fraction
    # --------------------------------

    top_ax = cbar.ax.twiny()
    top_ax.set_xlim(
        cbar.ax.get_xlim()
    )

    top_ax.set_xticks(z_match)
    top_ax.set_xticklabels(
        [f"{x:.1f}" for x in xhii_vals],
        fontsize=15
    )

    top_ax.set_xlabel(
        r"$\bar{x}_{\mathrm{H \ II}}$",
        fontsize=20,
        labelpad=10
    )

    # --------------------------------
    # Panel label
    # --------------------------------

    ax.text(
        0.03,
        0.95,
        r"(a)",
        transform=ax.transAxes,
        fontsize=28,
        ha="left",
        va="top"
    )

    # --------------------------------
    # Save / show
    # --------------------------------

    # plt.savefig(
    #     "VID_HI.png",
    #     dpi=300,
    #     bbox_inches="tight"
    # )

    plt.tight_layout()
    plt.show()

# --------------------------------
# Run
# --------------------------------

plot_vid_from_csv(PS_files)

### (b) One-point statistics, z = 10-5  (one-stat.ipynb cell 9)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import (FixedLocator, FuncFormatter,
                              NullFormatter, NullLocator)

# =====================================
# Read data
# =====================================
df = load_stats("H", 156, 406)          # snapshots 156-406 (z = 10.0-5.0), as before
z  = df["z"].values

SERIES = [
    ("std",        r"$\sigma$ [mK]", "black",      "-",   2.8),
    ("skewness",   r"Skewness",      "crimson",    "--",  2.4),
    ("kurtosis",   r"Kurtosis",      "seagreen",   "-.",  2.4),
    ("bimodality", r"BC",            "dodgerblue", ":",   2.6),
]

# =====================================
# Figure  (same size as panel a)
# =====================================
fig, ax = plt.subplots(figsize=(8, 6))

LINTHRESH = 1.0
YLO, YHI  = 0.1, 2000

# ---- shade the linear region (behind everything) ------------------
ax.axhspan(YLO, LINTHRESH, color="0.90", zorder=0, lw=0)
ax.axhline(LINTHRESH, color="0.45", lw=1.4, ls="-", zorder=1)

# ---- curves -------------------------------------------------------
for col, lab, c, ls, lw in SERIES:
    if col not in df.columns:
        print(f"[skip] {col} not in CSV")
        continue
    ax.plot(z, df[col].to_numpy(float), color=c, ls=ls, lw=lw,
            label=lab, zorder=3)

# ---- BC = 5/9 bimodality threshold --------------------------------
ax.axhline(5/9, color="dodgerblue", lw=1.0, ls=(0, (1, 3)),
           alpha=0.8, zorder=2)
ax.text(0.02, 5.5/9, r"BC $=5/9$", transform=ax.get_yaxis_transform(),
        color="dodgerblue", fontsize=15, ha="left", va="bottom", zorder=4)

# ---- scale + limits ----------------------------------------------
ax.set_yscale("symlog", linthresh=LINTHRESH, linscale=1.5)
ax.set_ylim(YLO, YHI)
ax.set_xlim(z.max(), z.min())

# ---- linear / log region annotations ------------------------------
ax.text(0.985, 0.30, "linear", transform=ax.get_yaxis_transform(),
        color="0.35", fontsize=17, ha="right", va="center",
         zorder=4)
ax.text(0.985, 3.0, "log", transform=ax.get_yaxis_transform(),
        color="0.35", fontsize=17, ha="right", va="center",
         zorder=4)

# ---- ticks --------------------------------------------------------
def _yfmt(v, _):
    if v < 1:
        return f"{v:g}"              # 0.1, 0.4, 0.6, 0.8
    if v == 1:
        return "1"
    if v == 10:
        return "10"
    e = int(round(np.log10(v)))
    return rf"$10^{{{e}}}$"          # 10^2, 10^3

major = [0.1, 0.4, 0.7, 1, 10, 100, 1000]
ax.yaxis.set_major_locator(FixedLocator(major))
ax.yaxis.set_major_formatter(FuncFormatter(_yfmt))
ax.yaxis.set_minor_locator(NullLocator())

ax.set_xlabel(r"$z$", fontsize=26)
ax.set_ylabel("VID statistic", fontsize=24)
ax.tick_params(which="both", direction="in", top=False, right=True,
               labelsize=20)

# ---- panel label + legend beneath it ------------------------------
ax.text(0.03, 0.95, r"(b)", transform=ax.transAxes,
        fontsize=28, ha="left", va="top", zorder=5)

ax.legend(fontsize=17, frameon=False, loc="upper left",
          bbox_to_anchor=(0.02, 0.87), ncol=2, columnspacing=1.4)

# ---- top axis: ionised fraction -----------------------------------
xhii_vals = np.array([0.90,     0.70,     0.50,     0.30,     0.10    ])
z_match   = np.array([5.826679, 6.342011, 6.872631, 7.624910, 9.170129])

top = ax.twiny()
top.set_xlim(ax.get_xlim())
top.set_xticks(z_match)
top.set_xticklabels([f"{x:.1f}" for x in xhii_vals], fontsize=18)
top.set_xlabel(r"$\bar{x}_{\mathrm{H\,II}}$", fontsize=22, labelpad=10)
top.tick_params(which="major", direction="in", labelsize=15)
top.xaxis.set_minor_locator(NullLocator())

plt.tight_layout()
# plt.savefig("VID_statistics_overlay.png", dpi=300, bbox_inches="tight")
plt.show()

## Helium (3.46 cm)


### (a) VID at <x_HeIII> = 0.1 ... 0.9  (one-stat.ipynb cell 31)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from matplotlib.ticker import NullLocator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --------------------------------
# Publication-quality style
# --------------------------------

mpl.rcParams.update({
    "font.size": 16,
    "axes.labelsize": 20,
    "axes.titlesize": 20,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "axes.linewidth": 2.0,
    "xtick.major.width": 1.8,
    "ytick.major.width": 1.8,
    "xtick.minor.width": 1.2,
    "ytick.minor.width": 1.2,
    "xtick.major.size": 7,
    "ytick.major.size": 7,
    "xtick.minor.size": 4,
    "ytick.minor.size": 4,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# --------------------------------
# Input files
# --------------------------------

# the snapshots where the volume-weighted <x_HeIII> = 0.1 ... 0.9 (check_ionization_history.py)
PS_files = [f"{MAPS['He']}/PS_{s:03d}.out" for s in (406, 450, 471, 487, 500, 514, 529, 558, 596)]


# --------------------------------
# Main function
# --------------------------------

def plot_vid_from_csv(PS_files):

    fig, ax = plt.subplots(figsize=(8, 6))

    zs, snaps = [], []

    for PS_file in PS_files:
        snaps.append(int(PS_file.split("_")[-1].split(".")[0]))
        zs.append(get_redshift(PS_file))

    zs = np.array(zs)

    # --------------------------------
    # Colormap
    # --------------------------------

    cmap = plt.get_cmap("viridis_r")
    norm = mpl.colors.Normalize(vmin=zs.min(), vmax=zs.max())

    # --------------------------------
    # Plot curves
    # --------------------------------

    for snap, z in zip(snaps, zs):

        filename = f"{VID['He']}/VID_He_{snap:03d}.csv"
        Tb, vid  = load_vid_csv(filename)

        vid = np.where(vid > 0, vid, 1e-20)

        ax.plot(
            Tb,
            vid,
            color=cmap(norm(z)),
            lw=4,
            alpha=0.8
        )

    # --------------------------------
    # Axis scales
    # --------------------------------

    ax.set_xscale("symlog", linthresh=1.0e-4) #0.01
    ax.set_yscale("log")

    # --------------------------------
    # Labels
    # --------------------------------

    ax.set_xlabel(
        r"$\delta T_b^{\mathrm{He}}\ [\mu{\rm K}]$", fontsize=24
    )

    ax.set_ylabel(
        r"$\delta T_b^{\mathrm{He}}$ PDF", fontsize=24
    )

    # --------------------------------
    # Limits
    # --------------------------------

    ax.set_xlim(0, 10)
    ax.set_ylim(1e-6, 1.0e5)

    # --------------------------------
    # Tick style
    # --------------------------------

    ax.tick_params(
        which="both",
        direction="in",
        top=True,
        right=True,
        labelsize=20
    )

    ax.xaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_locator(NullLocator())
    from matplotlib.ticker import FixedLocator, FuncFormatter

    xticks = [0, 1.0e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1]
    ax.xaxis.set_major_locator(FixedLocator(xticks))

    def _xfmt(v, _):
        if v == 0:
            return "0"
        return rf"$10^{{{int(round(np.log10(v)))}}}$"

    ax.xaxis.set_major_formatter(FuncFormatter(_xfmt))

    # --------------------------------
    # Embedded horizontal colorbar
    # --------------------------------

    sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cax = inset_axes(
        ax,
        width="45%",
        height="4%",
        loc="lower left",
        borderpad=4.0
    )

    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")

    cbar.set_label(r"$z$", fontsize=20)
    cbar.ax.tick_params(labelsize=15)

    # --------------------------------
    # Exact z <-> x_HeIII mapping
    # --------------------------------

    xhii_vals = np.array([0.90,     0.70,     0.50,     0.30,     0.20,     0.10    ])
    z_match   = np.array([3.439090, 3.746897, 4.011606, 4.299100, 4.517370, 4.999257])

    cbar.set_ticks(z_match)
    cbar.set_ticklabels([f"{z:.1f}" for z in z_match])

    # high-z on left
    cbar.ax.invert_xaxis()

    # --------------------------------
    # Top axis = ionized fraction
    # --------------------------------

    top_ax = cbar.ax.twiny()
    top_ax.set_xlim(cbar.ax.get_xlim())

    top_ax.set_xticks(z_match)
    top_ax.set_xticklabels([f"{x:.1f}" for x in xhii_vals], fontsize=15)

    top_ax.set_xlabel(
        r"$\bar{x}_{\mathrm{He\,III}}$",
        fontsize=20,
        labelpad=10
    )

    top_ax.xaxis.set_minor_locator(NullLocator())

    # --------------------------------
    # Panel label
    # --------------------------------

    ax.text(
        0.03,
        0.95,
        r"(a)",
        transform=ax.transAxes,
        fontsize=28,
        ha="left",
        va="top"
    )

    plt.tight_layout()
    # plt.savefig("VID_He.png", dpi=300, bbox_inches="tight")
    plt.show()


# --------------------------------
# Run
# --------------------------------

plot_vid_from_csv(PS_files)

### (b) One-point statistics, z = 6.1-3  (one-stat.ipynb cell 32)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import (FixedLocator, FuncFormatter,
                              NullFormatter, NullLocator)

# =====================================
# Read data
# =====================================
df = load_stats("He", 318, 705)         # snapshots 318-705 (z = 6.1-3.0), as before
z  = df["z"].values

SERIES = [
    ("std",        r"$100\,\sigma$ [$\mu$K]", "black",      "-",   2.8),
    ("skewness",   r"Skewness",               "crimson",    "--",  2.4),
    ("kurtosis",   r"Kurtosis",               "seagreen",   "-.",  2.4),
    ("bimodality", r"BC",                     "dodgerblue", ":",   2.6),
]
# =====================================
# Figure  (same size as panel a)
# =====================================
fig, ax = plt.subplots(figsize=(8, 6))

LINTHRESH = 1.0
YLO, YHI  = 0.1, 2000

# ---- shade the linear region (behind everything) ------------------
ax.axhspan(YLO, LINTHRESH, color="0.90", zorder=0, lw=0)
ax.axhline(LINTHRESH, color="0.45", lw=1.4, ls="-", zorder=1)

# ---- curves -------------------------------------------------------

SCALE = {"std": 100.0}          # column -> multiplicative factor

for col, lab, c, ls, lw in SERIES:
    if col not in df.columns:
        print(f"[skip] {col} not in CSV")
        continue
    y = df[col].to_numpy(float) * SCALE.get(col, 1.0)
    ax.plot(z, y, color=c, ls=ls, lw=lw, label=lab, zorder=3)

# ---- BC = 5/9 bimodality threshold --------------------------------
ax.axhline(5/9, color="dodgerblue", lw=1.0, ls=(0, (1, 3)),
           alpha=0.8, zorder=2)
ax.text(0.02, 5.5/9, r"BC $=5/9$", transform=ax.get_yaxis_transform(),
        color="dodgerblue", fontsize=15, ha="left", va="bottom", zorder=4)

# ---- scale + limits ----------------------------------------------
ax.set_yscale("symlog", linthresh=LINTHRESH, linscale=1.5)
ax.set_ylim(YLO, YHI)
ax.set_xlim(z.max(), z.min())

# ---- linear / log region annotations ------------------------------
ax.text(0.985, 0.30, "linear", transform=ax.get_yaxis_transform(),
        color="0.35", fontsize=17, ha="right", va="center",
        zorder=4)
ax.text(0.985, 3.0, "log", transform=ax.get_yaxis_transform(),
        color="0.35", fontsize=17, ha="right", va="center",
        zorder=4)

# ---- ticks --------------------------------------------------------
def _yfmt(v, _):
    if v < 1:
        return f"{v:g}"              # 0.1, 0.4, 0.7
    if v == 1:
        return "1"
    if v == 10:
        return "10"
    e = int(round(np.log10(v)))
    return rf"$10^{{{e}}}$"          # 10^2, 10^3

major = [0.1, 0.4, 0.7, 1, 10, 100, 1000]
ax.yaxis.set_major_locator(FixedLocator(major))
ax.yaxis.set_major_formatter(FuncFormatter(_yfmt))
ax.yaxis.set_minor_locator(NullLocator())

ax.set_xlabel(r"$z$", fontsize=26)
ax.set_ylabel("VID statistic", fontsize=24)
ax.tick_params(which="both", direction="in", top=False, right=True,
               labelsize=20)

# ---- panel label + legend beneath it ------------------------------
ax.text(0.03, 0.95, r"(b)", transform=ax.transAxes,
        fontsize=28, ha="left", va="top", zorder=5)

ax.legend(fontsize=17, frameon=False, loc="upper left",
          bbox_to_anchor=(0.02, 0.90), ncol=2, columnspacing=1.4)

# ---- top axis: He III fraction ------------------------------------
xhii_vals = np.array([0.90,     0.70,     0.50,     0.30,     0.10    ])
z_match   = np.array([3.439090, 3.746897, 4.011606, 4.299100, 4.999257])

keep      = (z_match >= z.min()) & (z_match <= z.max())
z_match   = z_match[keep]
xhii_vals = xhii_vals[keep]

top = ax.twiny()
top.set_xlim(ax.get_xlim())
top.set_xticks(z_match)
top.set_xticklabels([f"{x:.1f}" for x in xhii_vals], fontsize=18)
top.set_xlabel(r"$\bar{x}_{\mathrm{He\,III}}$", fontsize=22, labelpad=10)
top.tick_params(which="major", direction="in", labelsize=15)
top.xaxis.set_minor_locator(NullLocator())

plt.tight_layout()
# plt.savefig("VID_statistics_overlay_He.png", dpi=300, bbox_inches="tight")
plt.show()